In [1]:
# 키워드 기반 데이터 필터, 데이터프레임 생성, 타입 정리
from __future__ import annotations
from typing import Dict, Any, List
from collections.abc import Sequence, Iterable
import pandas as pd
import sys
import os
import psycopg2
from dotenv import load_dotenv
from psycopg2.extras import RealDictCursor
from functools import lru_cache
sys.path.append(os.path.abspath(".."))
from schemas import EventSearchRequest

COLUMN_MAP = {
    "title": "제목",
    "location": "장소",
    "organizer": "주최",
    "start_date": "시작 일시",
    "end_date": "종료 일시",
    "summary": "주제 요약",
    "event_type": "행사 성격",
    "keyword": "주요 키워드",
    "source": "출처",
    "registration_link": "등록 링크",
    "more_info_link": "상세 정보 링크",
    "is_not_free": "유료 여부",
}

DROPDOWN_FIELDS = ["organizer", "event_type", "location", "keyword"]
COMMA_SEPARATED_FIELDS = ["keyword"]

In [2]:
load_dotenv()

def get_db_connection():
    try:
        conn = psycopg2.connect(
            host=os.getenv("DB_HOST"),
            port=os.getenv("DB_PORT"),
            database=os.getenv("DB_NAME"),
            user=os.getenv("DB_USER"),
            password=os.getenv("DB_PASSWORD"),
        )
        print("[Debug] DB 연결 성공")
        return conn

    except Exception as e:
        print(f"[Error] DB 연결 실패: {e}")
        raise


class EventRepository:
    @lru_cache(maxsize=1)
    def load_events_dataframe(self) -> pd.DataFrame:
        query = """
            SELECT event_id, payload::jsonb -> 'source_row' AS source_row
            FROM public.events
            WHERE jsonb_typeof(payload::jsonb -> 'source_row') = 'object'
        """

        conn = get_db_connection()

        try:
            with conn.cursor(cursor_factory=RealDictCursor) as cur:
                cur.execute(query)
                rows: list[dict[str, Any]] = cur.fetchall()
        finally:
                conn.close()

        records: list[dict[str, Any]] = []

        for row in rows:
            source_row = dict(row["source_row"])
            source_row["event_id"] = str(row["event_id"])
            records.append(source_row)

        event_df = pd.DataFrame(records)
        print(f"[debug] DB조회 : {event_df.head()}")
        return event_df

    def clear_cache(self):
        self.load_events_dataframe.cache_clear()

In [3]:
repository = EventRepository()
event_df = repository.load_events_dataframe()

print(event_df.head())

[Debug] DB 연결 성공
[debug] DB조회 :    Index                          장소  \
0      1               Seattle, WA |   
1      2  한양대학교 국제관 6층 602호 (디지털강의실)   
2      3                   서울 대한전기협회   
3      4                         미기재   
4      5                         미기재   

                                                  제목  \
0                              U.S. Women in Nuclear   
1                 2026년 iTRS 몬테칼로 이론 및 실무교육(MCNP) 안내   
2                                          일반기계 공인검사   
3  Webinar: NEA–AFCONE Engagement Series with Afr...   
4                              가공배전 교육 (30일/240H) 94   

                           주최                               출처  \
0                   한국원자력산업협회                KAIF 원자력계 일정 AJAX   
1                     한국원자력학회         KNS news 게시판 제목 순회·본문 확인   
2                대한전기협회 KEPIC                      KEPIC 교육 일정   
3  OECD Nuclear Energy Agency  OECD NEA generated.Event search   
4              대한전기협회 전력기술교육원                   KEA 전력기술교육원 

In [4]:
def get_filter_options_data(df: pd.DataFrame) -> Dict[str, List[str]]:
    """DataFrame에서 필터 드롭다운을 위한 유니크 값들 추출"""
    result = {f"{field}s": [] for field in DROPDOWN_FIELDS}
    if df.empty:
        return result

    for field in DROPDOWN_FIELDS:
        plural_key = f"{field}s"
        db_column = COLUMN_MAP.get(field)
        
        if db_column and db_column in df.columns:
            if field in COMMA_SEPARATED_FIELDS:
                values_set = set()
                for string_val in df[db_column].dropna():
                    values_set.update([v.strip() for v in str(string_val).split(",") if v.strip()])
                result[plural_key] = sorted(list(values_set))
            else:
                result[plural_key] = sorted(df[db_column].dropna().unique().tolist())
    
    print(f"\n\n[DEBUG] 필터 옵션 드롭다운값 : {result} \n")
    return result

In [5]:
get_filter_options_data(event_df)



[DEBUG] 필터 옵션 드롭다운값 : {'organizers': ['IAEA', 'IEA; Government of Austria', 'IEA; International Emissions Trading Association (IETA); EPRI', 'IEA; Republic of Kenya; African Union Commission; African Development Bank; U.S. Department of Energy; Norway', 'IEA; South Africa Department of Electricity and Energy', 'IEEE PES', 'IEEE PES (재정 또는 기술 후원)', 'International Energy Agency (IEA)', 'OECD Nuclear Energy Agency', '대한전기협회', '대한전기협회 KEPIC', '대한전기협회 전력기술교육원', '한국원자력산업협회', '한국원자력학회'], 'event_types': ['Conference', 'KEC 교육', 'KEPIC 교육', 'Report launch', 'Training', 'Workshop', '공개 행사', '공지에서 확인된 행사', '교육·튜토리얼', '국제학술대회', '기술회의', '세미나·행사', '워크숍', '웨비나', '위원회 회의', '전력기술교육원 교육', '컨퍼런스', '학술대회·행사', '행사·교육'], 'locations': ['Alexandria, Virginia, USA', 'Amman, Jordan', 'Atlanta, Georgia, USA', 'Avignon, France |', 'Bangkok, Thailand', 'Beijing, China', 'Bogotá, Colombia', 'Bruges, Belgium', 'Budapest, Hungary', 'Chicago, IL |', 'Ciudad Real, Spain', 'College Station, TX, USA', 'College Station,

{'organizers': ['IAEA',
  'IEA; Government of Austria',
  'IEA; International Emissions Trading Association (IETA); EPRI',
  'IEA; Republic of Kenya; African Union Commission; African Development Bank; U.S. Department of Energy; Norway',
  'IEA; South Africa Department of Electricity and Energy',
  'IEEE PES',
  'IEEE PES (재정 또는 기술 후원)',
  'International Energy Agency (IEA)',
  'OECD Nuclear Energy Agency',
  '대한전기협회',
  '대한전기협회 KEPIC',
  '대한전기협회 전력기술교육원',
  '한국원자력산업협회',
  '한국원자력학회'],
 'event_types': ['Conference',
  'KEC 교육',
  'KEPIC 교육',
  'Report launch',
  'Training',
  'Workshop',
  '공개 행사',
  '공지에서 확인된 행사',
  '교육·튜토리얼',
  '국제학술대회',
  '기술회의',
  '세미나·행사',
  '워크숍',
  '웨비나',
  '위원회 회의',
  '전력기술교육원 교육',
  '컨퍼런스',
  '학술대회·행사',
  '행사·교육'],
 'locations': ['Alexandria, Virginia, USA',
  'Amman, Jordan',
  'Atlanta, Georgia, USA',
  'Avignon, France |',
  'Bangkok, Thailand',
  'Beijing, China',
  'Bogotá, Colombia',
  'Bruges, Belgium',
  'Budapest, Hungary',
  'Chicago, IL |',
  'Ciudad

In [6]:
def _apply_isin_filter(df: pd.DataFrame, column_name: str, values: List[str]) -> pd.DataFrame:
    """리스트에 정확히 일치하는 값을 필터링"""
    if values and column_name in df.columns:
        return df[df[column_name].isin(values)]
    return df

In [7]:
def _apply_contains_filter(df: pd.DataFrame, column_name: str, search_values: List[str]) -> pd.DataFrame:
    """값 중 하나라도 포함되어 있으면 필터링"""
    if search_values and column_name in df.columns:
        return df[df[column_name].apply(
            lambda x: any(v.lower() in str(x).lower() for v in search_values)
        )]
    return df

In [19]:
def _apply_date_filter(df: pd.DataFrame, start_col: str, end_col: str, start_date: str, end_date: str) -> pd.DataFrame:
    """시작 일시와 종료 일시를 기준으로 날짜 범위를 필터링합니다."""
    def _extract_and_parse_dates(series: pd.Series) -> pd.Series:
        extracted_dates = series.astype(str).str.extract(r'(\d{4}-\d{2}-\d{2})')[0]
        return pd.to_datetime(extracted_dates, errors='coerce')

    if start_date and start_col in df.columns:
        df = df[_extract_and_parse_dates(df[start_col]) >= pd.to_datetime(start_date)]
        
    if end_date and end_col in df.columns:
        df = df[_extract_and_parse_dates(df[end_col]) <= pd.to_datetime(end_date)]
        
    return df

In [9]:
def _map_row_to_event(row: pd.Series) -> Dict[str, Any]:
    """API 응답 스펙으로 변환"""
    event_item = {"event_id": str(row.get("event_id", ""))}

    for en_key, kr_key in COLUMN_MAP.items():
        if en_key not in COMMA_SEPARATED_FIELDS:
            event_item[en_key] = str(row.get(kr_key, ""))

    for field in COMMA_SEPARATED_FIELDS:
        plural_key = f"{field}s"
        kr_col = COLUMN_MAP.get(field)
        if kr_col and pd.notna(row.get(kr_col)):
            event_item[plural_key] = [v.strip() for v in str(row.get(kr_col, "")).split(",") if v.strip()]
        else:
            event_item[plural_key] = []

    return event_item

In [11]:
def filter_and_map_events(df: pd.DataFrame, request: EventSearchRequest) -> List[Dict[str, Any]]:
    if df.empty:
        return []

    filtered_df = df.copy()

    for field in DROPDOWN_FIELDS:
        plural_key = f"{field}s"
        req_values = getattr(request, plural_key, [])
        db_column = COLUMN_MAP.get(field)

        if req_values and db_column and db_column in filtered_df.columns:
            if field in COMMA_SEPARATED_FIELDS:
                filtered_df = _apply_contains_filter(filtered_df, db_column, req_values)
            else:
                filtered_df = _apply_isin_filter(filtered_df, db_column, req_values)

    #날짜 필터
    start_col = COLUMN_MAP.get("start_date")
    end_col = COLUMN_MAP.get("end_date")
    if start_col and end_col:
        filtered_df = _apply_date_filter(filtered_df, start_col, end_col, request.start_date, request.end_date)

    return [_map_row_to_event(row) for _, row in filtered_df.iterrows()]

In [17]:
import json
def print_test_result(test_name: str, request: EventSearchRequest):
    print(f"========================================")
    print(f"테스트명: {test_name}")
    print(f"요청 파라미터: {request.model_dump(exclude_none=True)}")
    
    # 캐시된(또는 로드된) df가 있다고 가정 (사전에 df = repository.get_cached_dataframe() 실행 필요)
    results = filter_and_map_events(event_df, request)
    
    print(f">>  검색된 결과 수: {len(results)}개")
    if results:
        # 첫 번째 결과만 살짝 미리보기 (가독성을 위해 json.dumps 사용)
        print(f">>>> 첫 번째 결과 미리보기:\n{json.dumps(results[0], indent=2, ensure_ascii=False)}")
    print(f"========================================\n")

In [14]:
# Test 1: 아무 조건도 없는 텅 빈 요청 (전체 데이터가 반환되어야 함)
req_empty = EventSearchRequest()

# Test 2: 단일 카테고리 다중 선택 (장소가 'Osaka, Japan' 이거나 'Seoul, Korea'인 행사)
req_location_only = EventSearchRequest(
    locations=["Osaka, Japan", "Seoul, Korea"]
)

# Test 3: 카테고리 간 교집합 AND 테스트 (주최가 'IEEE PES' 이면서, 행사가 '컨퍼런스'인 것)
req_and_condition = EventSearchRequest(
    organizers=["IEEE PES", "IEEE PES (재정 또는 기술 후원)"],
    event_types=["컨퍼런스"]
)

# Test 4: 키워드 부분 일치 테스트 ('전력' 또는 '에너지'가 주요 키워드에 포함된 것)
req_keywords = EventSearchRequest(
    keywords=["전력", "에너지"]
)

# Test 5: 날짜 필터 테스트 (특정 날짜 사이에 열리는 행사)
req_dates = EventSearchRequest(
    start_date="2026-08-01",
    end_date="2026-08-31"
)

# Test 6: 종합 복합 필터 (실제 서비스에서 가장 하드하게 들어올 수 있는 요청)
req_complex = EventSearchRequest(
    event_types=["컨퍼런스", "세미나"],
    keywords=["국제", "tech"],
    locations=["Osaka, Japan"],
    start_date="2026-08-15"
    # end_date는 안 보냈으므로, 8월 15일 '이후'의 행사만 검색되어야 함
)

In [20]:
print_test_result("Test 1: 전체 행사 조회", req_empty)
print_test_result("Test 2: 장소 OR 필터", req_location_only)
print_test_result("Test 3: 주최 & 성격 AND 필터", req_and_condition)
print_test_result("Test 4: 키워드 부분 일치 필터", req_keywords)
print_test_result("Test 5: 날짜 범위 필터", req_dates)
print_test_result("Test 6: 종합 복합 필터", req_complex)

테스트명: Test 1: 전체 행사 조회
요청 파라미터: {'organizers': [], 'event_types': [], 'keywords': [], 'locations': []}
>>  검색된 결과 수: 222개
>>>> 첫 번째 결과 미리보기:
{
  "event_id": "92389c81-3d25-4a11-8ec4-3ec7cf5cc803",
  "title": "U.S. Women in Nuclear",
  "location": "Seattle, WA |",
  "organizer": "한국원자력산업협회",
  "start_date": "2026-08-02 00:00:00 GMT (날짜 전용)",
  "end_date": "2026-08-05 23:59:59 GMT (날짜 전용)",
  "summary": "KAIF 원자력계 일정의 공개 행사",
  "event_type": "세미나·행사",
  "source": "KAIF 원자력계 일정 AJAX",
  "registration_link": "nan",
  "more_info_link": "https://www.nei.org/conferences#section1",
  "is_not_free": "확인 불가",
  "keywords": [
    "원자력산업",
    "KAIF"
  ]
}

테스트명: Test 2: 장소 OR 필터
요청 파라미터: {'organizers': [], 'event_types': [], 'keywords': [], 'locations': ['Osaka, Japan', 'Seoul, Korea']}
>>  검색된 결과 수: 3개
>>>> 첫 번째 결과 미리보기:
{
  "event_id": "7163057f-f206-4f87-b65a-aa8f2abdd22a",
  "title": "2026 5th International Conference on Power Systems and Electrical Technology (PSET)",
  "location": "Osaka, J